# ⛰ Core Lab 7 — Feeding the GPU
**A Billion Small Steps · Hardware for AI · Step 3 · Book sections 3.3–3.4 · Foothold lessons 3.3–3.4**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** bandwidth rising then levelling off as threads increase; strided access several times slower than coalesced.

| Part | What you do | Book |
|---|---|---|
| A | Add threads until the waiting is hidden | 3.3 |
| B | Bandwidth: coalesced vs strided | 3.4 |

⏱️ About 2 hours. 📘 Textbook: Hwu, Kirk & El Hajj Ch 4–5 · Hamacher §12.1, Ch 8. Course site: https://aishugithub.github.io/course_GPU/

🥾 Side Trail for this Step (map a job onto a GPU: plan vs reality) is in the **Step 3 Side Trails** notebook.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

In [ ]:
sms = torch.cuda.get_device_properties(0).multi_processor_count   # how many SMs this GPU has (Core Lab 6 found it too)
print(f"This GPU has {sms} SMs")                                  # show it (a T4 has 40)

## Part A · Hiding Latency  *(Section 3.3)*

A copy kernel where each thread copies elements in a loop (a "grid-stride loop"). We launch **more and more threads** for the same 256 MB copy.
With few threads, each warp mostly waits for memory. With many, the scheduler always has a ready warp.

In [ ]:
copy_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void copy(const float *x, float *y, int n) {   // copy x into y, n numbers
    // the total number of threads launched
    int stride = gridDim.x * blockDim.x;
    // each thread starts at its own index, copies, then jumps ahead by "stride" until it passes the end
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        y[i] = x[i];   // copy one element
}
'''
copy_k = cp.RawKernel(copy_src, "copy")   # compile the kernel
# 6.7 crore floats = 256 MB to copy; the source and destination both live in GPU memory
n = 1 << 26
xs, ys = cp.random.rand(n, dtype=cp.float32), cp.empty(n, dtype=cp.float32)   # source and destination on the GPU
total_threads, bandwidth = [], []   # the results go into these lists
# launch more and more blocks of 256 threads for the same copy
for blocks in [1, 4, 16, 40, 80, 160, 320, 640, 1280]:
    ms = cuda_time_ms(lambda: copy_k((blocks,), (256,), (xs, ys, cp.int32(n))), repeats=3)   # time the copy with this many blocks
    # bytes moved (read + write, 4 bytes each) ÷ seconds ÷ 1e9 = GB/s
    gbs = 2 * n * 4 / (ms / 1e3) / 1e9
    total_threads.append(blocks * 256); bandwidth.append(gbs)   # keep the results for the chart
    # warps per SM = threads ÷ 32 threads per warp ÷ the number of SMs
    print(f"{blocks*256:>8,d} threads ({blocks*256/32/sms:6.1f} warps per SM): {gbs:6.1f} GB/s")

In [ ]:
plt.figure(figsize=(6, 3.5))                                  # a new chart
plt.semilogx(total_threads, bandwidth, "o-")                  # log scale on the x axis only
plt.xlabel("threads launched"); plt.ylabel("GB/s")            # axis labels
plt.grid(alpha=0.3, which="both")                             # faint grid
plt.title("More warps → memory latency hidden → bandwidth rises, then levels off")   # the chart's title
plt.show()                                                    # draw it

### ✏️ YOUR TURN (A)
Roughly how many warps per SM did it take before the bandwidth levelled off? Connect this to the tea master in Section 3.3.

*Your answer:*

## Part B · Bandwidth: Coalesced vs Strided  *(Section 3.4)*

First, the best-case bandwidth: copy a big tensor inside GPU memory with PyTorch.

In [ ]:
# 13.4 crore floats = 512 MB; .clone() copies it inside GPU memory: the best case for bandwidth
big = torch.rand(1 << 27, device=device)
t = timeit(lambda: big.clone(), repeats=5)   # time one copy
# bandwidth = bytes read + bytes written ÷ time
best_bw = 2 * big.numel() * 4 / t
print(f"Copy 512 MB inside the GPU: {t*1e3:.1f} ms → {best_bw / 1e9:.0f} GB/s  (a T4's published peak: 320 GB/s)")   # show the bandwidth

Now a kernel where thread i reads element `(i × stride) % n`. With stride 1, neighbouring threads read neighbouring floats (coalesced). With bigger strides, the threads of a warp land in different pieces of memory.

In [ ]:
stride_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void strided(const float *x, float *y, int n, int stride) {   // copy with a gap of "stride" between the reads
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's index
    // thread i reads position i × stride (wrapped round with % n); long long avoids overflow
    if (i < n) y[i] = x[(long long)i * stride % n];
}
'''
strided = cp.RawKernel(stride_src, "strided")   # compile the kernel
# about 3.4 crore floats = 128 MB, one thread per element, 256 threads per block
n = 1 << 25
xs, ys = cp.random.rand(n, dtype=cp.float32), cp.empty(n, dtype=cp.float32)   # source and destination on the GPU
blocks = (n + 255) // 256   # enough blocks of 256 threads to cover n (rounded up)
stride_ms = {}   # stride → time in ms
# stride 1: neighbours read neighbours; stride 32: every thread of a warp lands in a different chunk
for s in [1, 2, 8, 32]:
    stride_ms[s] = cuda_time_ms(lambda: strided((blocks,), (256,), (xs, ys, cp.int32(n), cp.int32(s))))   # time the kernel with this stride
    print(f"stride {s:2d}: {stride_ms[s]:6.2f} ms   useful bandwidth {2 * n * 4 / (stride_ms[s] / 1e3) / 1e9:6.1f} GB/s")   # show the time and the useful bandwidth

### ✏️ YOUR TURN (B)
How many times slower was stride 32 than stride 1? Why doesn't stride 2 hurt as much?

*Your answer:*

---
## 🎓 Wrap-up
- Memory is slow; the GPU hides the wait by switching between many warps. Too few threads leave it idle.
- Neighbouring threads should read neighbouring addresses (coalesced): strided reads waste most of each memory transaction.
- **Next — Core Lab 8: Hand a Job to the GPU.** Allocate, copy, launch your own kernel, copy back.